# Pre-preparation

Mount Drive

In [ ]:
from google.colab import drive
drive.mount("gdrive")

Install piq for LPIPS

In [ ]:
!pip install piq
! pip install wandb

Install W&B and login

In [ ]:
import wandb
wandb.login()

In [ ]:
# import sys

# class NullWandB:
#     """
#     A Null Object that mimics the wandb API but does nothing.
#     """
#     def __init__(self, *args, **kwargs):
#         self.run = self # mimic the .run attribute
#         self.config = self # mimic the .config attribute

#     def init(self, *args, **kwargs): return self
#     def log(self, *args, **kwargs): pass
#     def finish(self, *args, **kwargs): pass
#     def watch(self, *args, **kwargs): pass
#     def config_update(self, *args, **kwargs): pass

#     def Image(self, data, *args, **kwargs):
#         # Return the raw data or a placeholder string so list comprehensions don't break
#         return "WandB Disabled: Image Placeholder"
#     def __getattr__(self, name):
#         # This magic method handles any other method calls (like .watch, .save, etc.)
#         # and returns self so you can chain calls safely.
#         return self
#     def __call__(self, *args, **kwargs):
#         # Allows the object itself to be called if needed
#         return self


# # Configuration Flag
# USE_WANDB = True

# try:
#     if not USE_WANDB:
#         raise ImportError("User disabled WandB") # Force the except block

#     import wandb
#     print("WandB imported successfully.")

# except ImportError:
#     print("WandB not found or disabled. Using MockWandB (No-Op mode).")
#     wandb = MockWandB() # <--- SWAP IN THE MOCK


Configure the project & save directory path

In [ ]:
project_path = '/content/gdrive/MyDrive/New_Linearized_Flow_Matching_Files/Linearizer'
SAVE_DIR_BASE = '/content/gdrive/MyDrive/NLFM_models'

# project_path = '/content/gdrive/MyDrive/NLFM/Linearizer'
# SAVE_DIR_BASE = '/content/gdrive/MyDrive/NLFM/NLFM_models'

# Main Code

## 1. Preparation

In [ ]:
if project_path not in sys.path:
    sys.path.append(project_path)


os.chdir(project_path) # Change the working directory to that folder

print(f"Current Working Directory: {os.getcwd()}")
print("Files in this directory:")
print(os.listdir('.'))

Current Working Directory: /content/gdrive/MyDrive/New_Linearized_Flow_Matching_Files/Linearizer
Files in this directory:
['linearizer.py', 'new_linearized_flow_matching.py', 'common', 'one_step', '__pycache__', 'data']


In [ ]:
import sys
import os


MODEL_INDEX = 47
MODEL_NAME = f'model{MODEL_INDEX}'
SAVE_DIR = os.path.join(SAVE_DIR_BASE, MODEL_NAME)
SAVE_DIR_RAW = os.path.join(SAVE_DIR, 'RAW')
SAVE_DIR_EMA = os.path.join(SAVE_DIR, 'EMA')
if not os.path.exists(SAVE_DIR_RAW):
    os.makedirs(SAVE_DIR_RAW)
if not os.path.exists(SAVE_DIR_EMA):
    os.makedirs(SAVE_DIR_EMA)


Hyper-parameter configuration

In [ ]:
# --- Configuration ---
DATASET = 'mnist'
IMG_SIZE = 32
IN_CHANNELS = 1
BATCH_SIZE = 64 #32 #64 #32 #64
VAL_BATCH_SIZE = 64 #32 #64 #32 #64

MODEL_CHANNELS = 16 #32    # U-Net capacity
NUM_LAYERS_G = 3 #4 #3 #4       # Depth of g

LR = 1e-4 #1e-3 #1e-4 #1e-2 #1e-4
EPOCHS = 10 #20
EVAL_INTERVAL = 1   # Print samples every N epochs
EMA_DECAY = 0.9999 #0.999

# IS_GRADIENT_CLIP = True
GRADIENT_CLIP_THRESOLD = 3.0 #2.0 #1.0 #10.0

# Parameter for whether we use the inv_t, and what t will be - the t of the gt, randomly sampled, constant like 0/1/0.5
INT_T_CONF = 'False' #['False', 'T', 'Random', '1']


LAMBDA_FM_L2 = 0.0 #1.0 #2.5 #1.0
LAMBDA_FM_LPIPS = 0.0 #1e-1 #1e-2
LAMBDA_ISO = 0.0 #1e-4 #1e-3 #1e-1 #1e-3 #0.0 #1e-3 #1e-4 #1e-3 #1e-4 #1e-2
LAMBDA_FROB = 0.0 #1e-3 #0.0 #1e-3
LAMBDA_SPEC = 0.0 #1e-3 #1e-1 #1e-3
LAMBDA_VEL = 0.0 #1e-3
LAMBDA_TARGET_L2 = 1.0 #0.5 #1e-1 #1e-3 #0.0 #1e-3
LAMBDA_TARGET_LPIPS = 1e-1 #1e-2 #1e-3 #1e-1 #1e-5

LAMBDAS_DICT = {
    'FM_L2': LAMBDA_FM_L2,
    'FM_LPIPS': LAMBDA_FM_LPIPS,
    'ISO': LAMBDA_ISO,
    'FROB': LAMBDA_FROB,
    'SPEC': LAMBDA_SPEC,
    'VEL': LAMBDA_VEL,
    'TARGET_L2': LAMBDA_TARGET_L2,
    'TARGET_LPIPS': LAMBDA_TARGET_LPIPS
}

NUM_SAMPLING_STEPS = 100 # for dicrete and iterative sampling

NUM_SAMPLES = 16 # how many samples (per type) to show each time during training

INIT_FACTOR_A = 1e-2 # initialization factor for the noise A is initialized with

# maybe set name according to configuration

# hyper-parameters to maybe add:
# dropout, batch norm, weight init (for A and also for gt)
# p for the bipitite matching (though try other matching techniques)

Initialize a wandb run

In [ ]:
def wandb_init(
        wandb=wandb,
        project_name="linearized-flow-matching",
        model_name=MODEL_NAME,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        gradient_clip=GRADIENT_CLIP_THRESOLD,
        optimizer="AdamW",
        model_channels=MODEL_CHANNELS,
        num_layers_g=NUM_LAYERS_G,
        img_size=IMG_SIZE,
        in_channels=IN_CHANNELS,
        inv_t_conf=INT_T_CONF,
        ema_decay=EMA_DECAY,
        lambdas_dict=LAMBDAS_DICT,
    ):
    wandb_run = wandb.init(
        project=project_name,
        name=model_name,
        config={
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "gradient_clip": gradient_clip,
            "optimizer": optimizer,
            "model_channels": model_channels,
            "num_layers_g": num_layers_g
            "img_size": img_size,
            "in_channels": in_channels,
            "inv_t_conf": inv_t_conf,
            "ema_decay": ema_decay,
            **lambdas_dict
        }
    )

    return wandb_run

wandb_run = wandb_init() # Instantiate

Imports and device configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.linalg

import torchvision.utils as vutils

from scipy.optimize import linear_sum_assignment
from piq import LPIPS

sys.path.append(os.getcwd())  # Setup path to import existing project files

# from one_step.data.data_utils import get_data_loaders
from one_step.modules.linear_network import OneStepLinearModule
from one_step.modules.invertable_network_new import InvUnet, InverseUnet
from common.song__unet import creat_song_unet
from linearizer import Linearizer
from tqdm import tqdm

import multiprocessing
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


Set random seed

In [ ]:
def seed_everything(seed=42):
    """
    Locks the random seed for reproducibility.
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # These two lines ensure the GPU operations are deterministic
    # (might slightly slow down training, but worth it for debugging)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Global Seed set to {seed}")

seed_everything()

## 2. Classes

Exponential Moving Average

In [ ]:
import copy

class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.model = copy.deepcopy(model)
        self.model.eval() # EMA model is always in eval mode

        for param in self.model.parameters(): # We don't want to optimize the EMA parameters
            param.requires_grad = False

    def update(self, model): # Updates the EMA model weights based on the current model.
        with torch.no_grad():
            for ema_param, param in zip(self.model.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(param.data, alpha=1 - self.decay)

    def get_model(self):
        return self.model

The fixed matrix 'A'

In [ ]:
# --- 1. The Fixed Linear Operator A ---
class FixedLinearMatrix(OneStepLinearModule):
    def __init__(self, dim, init_factor=INIT_FACTOR_A):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(dim, dim) * init_factor)
        self.register_buffer('exp_A_cached', None)

    def forward(self, x, **kwargs):
        original_shape = x.shape
        if x.dim() > 2: x = x.view(x.shape[0], -1)
        out = x @ self.weight.t()
        return out.view(original_shape)

    def cache_exponential(self):
        with torch.no_grad():
            self.exp_A_cached = torch.linalg.matrix_exp(self.weight.t())

    def forward_exponential(self, x):
        if self.exp_A_cached is None: self.cache_exponential()
        original_shape = x.shape
        if x.dim() > 2: x = x.view(x.shape[0], -1)
        out = x @ self.exp_A_cached
        return out.view(original_shape)

    def get_lin_t(self, t):
        return self.weight.unsqueeze(0)

 The time-conditioned invertible NN 'g' (warpped with a linearizer)

In [ ]:
# ---- The Time-Varying Wrapper ---
class TimeVaryingGLinearizer(Linearizer):
    def __init__(self, g_net, fixed_A_net):
        # We pass g_net for both gx and gy because it handles time internally
        super().__init__(gx=g_net, linear_network=fixed_A_net, gy=g_net)

    def g(self, x, t):
        return self.net_gx(x, t=t)

    def g_inverse(self, z, t):
        return self.net_gx.inverse(z, t=t)

    def A(self, z):
        return self.linear_network(z)

Class for calculation of the isometry loss term

In [ ]:
class IsometryLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, g_x, g_0):
        """
        Calculates L_isometry = | ||g(x) - g(0)||^2 - ||x||^2 |

        Args:
            x: Input data [B, C, H, W]
            g_x: Latent representation of x [B, C, H, W]
            g_0: Latent representation of zero [B, C, H, W] (or [1, C, H, W])
        """
        # Flatten to [B, D] for norm calculation
        x_flat = x.view(x.shape[0], -1)
        g_x_flat = g_x.view(x.shape[0], -1)
        g_0_flat = g_0.view(g_0.shape[0], -1) # Handle broadcasting if g_0 is single sample

        # Calculate squared norms
        # Norm of input x
        norm_x_sq = torch.sum(x_flat ** 2, dim=1)

        # Norm of displacement in latent space (g(x) - g(0))
        # Note: g_0 might need broadcasting if it's calculated once per batch or once globally
        diff = g_x_flat - g_0_flat
        norm_g_diff_sq = torch.sum(diff ** 2, dim=1)

        # The Loss: Absolute difference between the squared norms
        loss = torch.abs(norm_g_diff_sq - norm_x_sq).mean()

        return loss


Minibatch pairing function, currently bipirtite matching with Lp norm

In [ ]:
@torch.no_grad()
def pair_batch(x0, x1, p=1):
    batch_size = x0.shape[0]
    x0_flat = x0.view(batch_size, -1) # flatten
    x1_flat = x1.view(batch_size, -1)
    cost_matrix = torch.cdist(x0_flat, x1_flat, p=p).cpu().numpy() # calculate bipirtite matching pairing where distance is L2

    # linear_sum_assignment finds the optimal non-overlapping pairs
    # row_ind corresponds to x0 indices, col_ind corresponds to x1 indices
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # create a new x0 tensor where the noise is sorted to match the order of x1
    # sort the rows of x0 based on where the algorithm told them to go (col_ind)
    sorted_x0 = torch.zeros_like(x0)
    for i in range(batch_size):
        sorted_x0[col_ind[i]] = x0[row_ind[i]]

    return sorted_x0

**The final flow matching model**, with methods for: loss calculation, sampling, diagnostics

In [ ]:
class TimeG_FlowMatcher:
    def __init__(
            self,
            linearizer,
            inv_t_conf=INT_T_CONF,
            lambdas_dict=LAMBDAS_DICT
            ):
        self.linearizer = linearizer
        self.iso_loss_fn = IsometryLoss()

        # self.lpips_fn = LPIPS(replace_pooling=True, reduction="mean") if lambda_FM_LPIPS > 0 else None
        self.lpips_fn = LPIPS(replace_pooling=True, reduction="mean")

        self.inv_t_conf = inv_t_conf



        self.lambda_FM_L2 = lambdas_dict['FM_L2']
        self.lambda_FM_LPIPS = lambdas_dict['FM_LPIPS']

        self.lambda_target_L2 = lambdas_dict['TARGET_L2']
        self.lambda_target_LPIPS = lambdas_dict['TARGET_LPIPS']

        self.lambda_iso = lambdas_dict['ISO']
        self.lambda_frob = lambdas_dict['FROB']
        self.lambda_spec = lambdas_dict['SPEC']
        self.lambda_vel = lambdas_dict['VEL']


    def _calc_fm_loss(self, velocity_pred, velocity_target, t, batch_size, device):
        if self.lambda_FM_L2 <= 0:
            return torch.tensor(0.0, device=device)

        if self.inv_t_conf == 'False':
            return ((velocity_pred - velocity_target) ** 2).mean()

        inv_t = None
        if self.inv_t_conf == '1':
            inv_t = torch.ones(batch_size, device=device)
        elif self.inv_t_conf == 'Random':
            inv_t = torch.rand(batch_size, device=device)
        else: # 'T'
            inv_t = t.clone()

        inv_velocity_pred = self.linearizer.g_inverse(velocity_pred, t=inv_t)
        inv_velocity_target = self.linearizer.g_inverse(velocity_target, t=inv_t)
        return ((inv_velocity_pred - inv_velocity_target) ** 2).mean()

    def _calc_lpips_fm_loss(self, x0, x1, velocity_pred, device):
        # Currently placeholder based on provided code
        return torch.tensor(0.0, device=device)
        # if self.lpips_fn is not None: # if self.lambda_FM_LPIPS > 0:
        #     lpips_fm_loss = torch.tensor(0.0, device=x1.device)
        #     if self.lambda_FM_LPIPS > 0:
        #         # Step A: Extrapolate the predicted x1 from the velocity
        #         # If your velocity is defined as v = x1 - x0, then:
        #         # x1_pred = x0 + inv_velocity_pred
        #         # (Adjust this formula if your velocity definition is different!)
        #         x1_pred = x0 + inv_velocity_pred

        #         # Step B: LPIPS expects 3-channel RGB images.
        #         # If using MNIST (1 channel), we must duplicate the channels.
        #         if x1.shape[1] == 1:
        #             x1_pred_rgb = x1_pred.repeat(1, 3, 1, 1)
        #             x1_target_rgb = x1.repeat(1, 3, 1, 1)
        #         else:
        #             x1_pred_rgb = x1_pred
        #             x1_target_rgb = x1

        #         # Step C: LPIPS expects values roughly in the [0, 1] range.
        #         # We clamp to prevent the VGG network from seeing wild extrapolated values
        #         lpips_fm_loss = self.lpips_fn(
        #             x1_pred_rgb.clamp(0, 1),
        #             x1_target_rgb.clamp(0, 1)
        #         )

    def _predict_target_x1(self, x0, t_start, batch_size, device):
        """Helper to generate hat_x_1 for target losses"""
        # t_0 = torch.zeros(batch_size, device=device)

        t_start = t_start.clone()
        z_0 = self.linearizer.g(x0, t_start)
        if isinstance(z_0, tuple): z_0 = z_0[0]

        # Matrix Exponential Jump
        A = next(self.linearizer.linear_network.parameters())
        exp_A = torch.matrix_exp(A)

        z_0_flat = z_0.view(batch_size, -1)
        z_1_flat = torch.matmul(exp_A, z_0_flat.T).T
        z_1_predicted = z_1_flat.view_as(x0)

        t_1 = torch.ones(batch_size, device=device)
        hat_x_1 = self.linearizer.g_inverse(z_1_predicted, t_1)
        if isinstance(hat_x_1, tuple): hat_x_1 = hat_x_1[0]

        return hat_x_1

    def _calc_target_l2_loss(self, hat_x_1, x1, device):
        if self.lambda_target_L2 > 0:
            return F.mse_loss(hat_x_1, x1)
        return torch.tensor(0.0, device=device)

    def _calc_target_lpips_loss(self, hat_x_1, x1, device):
        if self.lambda_target_LPIPS > 0:
            if x1.shape[1] == 1:
                hat_x_1_rgb = hat_x_1.repeat(1, 3, 1, 1)
                x1_rgb = x1.repeat(1, 3, 1, 1)
            else:
                hat_x_1_rgb = hat_x_1
                x1_rgb = x1

            return self.lpips_fn(
                hat_x_1_rgb.clamp(-1.0, 1.0),
                x1_rgb.clamp(-1.0, 1.0)
            )
        return torch.tensor(0.0, device=device)

    def _calc_frob_loss(self, matrix_A, device):
        if self.lambda_frob > 0:
            return torch.sum(matrix_A ** 2)
        return torch.tensor(0.0, device=device)

    def _calc_spectral_loss(self, matrix_A, device):
        if self.lambda_spec > 0:
            return torch.linalg.matrix_norm(matrix_A, ord=2)
        return torch.tensor(0.0, device=device)

    def _calc_vel_loss(self, velocity_pred, device):
        if self.lambda_vel > 0:
            return torch.sum(velocity_pred ** 2)
        return torch.tensor(0.0, device=device)

    def _calc_iso_loss(self, x_t, g_t_x_t, t, x1_shape, device):
        if self.lambda_iso > 0:
            zeros = torch.zeros(x1_shape, device=device)
            g_t_0 = self.linearizer.g(zeros, t=t)
            return self.iso_loss_fn(x_t, g_t_x_t, g_t_0)
        return torch.tensor(0.0, device=device)

    def training_losses(self, x1):
        batch_size = x1.shape[0]
        device = x1.device

        # --- Compute needed values for loss calculation ---
        x0 = torch.randn_like(x1) # sample gaussian noise ~ N(0,1)
        x0 = pair_batch(x0, x1) # re-pair the batch
        t = torch.rand(batch_size, device=device) # sample random values for time t, uniform distribution
        t0 = torch.zeros(batch_size, device=device) # t0 = 0
        t1 = torch.ones(batch_size, device=device) # t1 = 1
        g0_x0 = self.linearizer.g(x0, t=t0) # g0(x0)
        g1_x1 = self.linearizer.g(x1, t=t1) # g1(x1)

        # We pass xt through gt
        zt = (1 - t[:, None, None, None]) * g0_x0 + t[:, None, None, None] * g1_x1 # interpolation: zt = (1-t)*g0(x0) + t*g1(x1)
        xt = self.linearizer.g_inverse(zt, t=t) # xt = gt^-1(zt)
        gt_xt = self.linearizer.g(xt, t=t) # gt(xt) (should be equal to zt)

        velocity_pred = self.linearizer.A(gt_xt) # A*gt(xt)
        velocity_target = g1_x1 - g0_x0 # g1(x1) - g0(x0)

        matrix_A = self.linearizer.linear_network.weight # A

        # --- Calculate Loss Terms ---

        # FM Losses
        fm_loss = self._calc_fm_loss(velocity_pred, velocity_target, t, batch_size, device)
        lpips_fm_loss = self._calc_lpips_fm_loss(x0, x1, velocity_pred, device)

        # Target Losses
        target_L2_loss = torch.tensor(0.0, device=device)
        target_LPIPS_loss = torch.tensor(0.0, device=device)

        if self.lambda_target_L2 > 0 or self.lambda_target_LPIPS > 0:
            hat_x_1 = self._predict_target_x1(x0, batch_size, device)
            target_L2_loss = self._calc_target_l2_loss(hat_x_1, x1, t_start=t0, device)
            target_LPIPS_loss = self._calc_target_lpips_loss(hat_x_1, x1, t_start=t0, device)

        # Regularization Losses
        frob_loss = self._calc_frob_loss(matrix_A, device)
        spectral_loss = self._calc_spectral_loss(matrix_A, device)
        vel_loss = self._calc_vel_loss(velocity_pred, device)
        iso_loss = self._calc_iso_loss(xt, gt_xt, t, x1.shape, device)

        # --- Total Loss ---
        total_loss = (
            self.lambda_FM_L2 * fm_loss
            + (self.lambda_FM_LPIPS * lpips_fm_loss)
            + (self.lambda_target_L2 * target_L2_loss)
            + (self.lambda_target_LPIPS * target_LPIPS_loss)
            + (self.lambda_frob * frob_loss)
            + (self.lambda_spec * spectral_loss)
            + (self.lambda_vel * vel_loss)
            + (self.lambda_iso * iso_loss)
        )

        return {
            "loss/total": total_loss,
            "loss/fm": fm_loss.item(),
            "loss/lpips": lpips_fm_loss.item(),
            "loss/target_l2": target_L2_loss.item(),
            "loss/target_lpips": target_LPIPS_loss.item(),
            "loss/iso": iso_loss.item(),
            "loss/spectral": spectral_loss.item(),
            "loss/frob": frob_loss.item(),
            "loss/velocity": vel_loss.item()
        }



    @torch.no_grad()
    def _test_invertibility(self, x, t):
        """
        Test 1: Round-trip Invertibility (x -> z -> x_rec)
        """
        # Forward
        z = self.linearizer.g(x, t) # z = gt(x)
        if isinstance(z, tuple): z = z[0]

        # Inverse
        x_rec = self.linearizer.g_inverse(z, t) # x_rec =  gt^-1(z) = gt^-1(gt(x))
        if isinstance(x_rec, tuple): x_rec = x_rec[0]

        # Metrics
        mse = F.mse_loss(x_rec, x).item()
        max_err = torch.max(torch.abs(x_rec - x)).item()

        return {"round_trip_mse": mse, "round_trip_max_error": max_err}

    @torch.no_grad()
    def _test_latent_space(self, z):
        """
        Test 2: Latent Space Statistics (Check for exploding values)
        """
        return {
            "latent_min": z.min().item(),
            "latent_max": z.max().item(),
            "latent_std": z.std().item()
        }

    @torch.no_grad()
    def _test_time_sensitivity(self, x):
        """
        Test 3: Time Embedding Sensitivity
        Checks if g(x, t=0) is significantly different from g(x, t=1)
        """
        batch_size = x.shape[0]
        device = x.device

        t0 = torch.zeros(batch_size, device=device)
        t1 = torch.ones(batch_size, device=device)

        z0 = self.linearizer.g(x, t0)
        z1 = self.linearizer.g(x, t1)

        if isinstance(z0, tuple): z0 = z0[0]
        if isinstance(z1, tuple): z1 = z1[0]

        diff = torch.mean(torch.abs(z0 - z1)).item()
        return {"time_sensitivity_diff": diff}

    @torch.no_grad()
    def _test_matrix_stability(self, x_noise, device):
        """
        Test 4: Matrix Exponential Stability
        Checks if e^A * z0 produces reasonable values compared to a standard normal
        """
        batch_size = x_noise.shape[0]
        t0 = torch.zeros(batch_size, device=device)

        # Encode noise at t=0
        z0 = self.linearizer.g(x_noise, t0)
        if isinstance(z0, tuple): z0 = z0[0]

        # Calculate e^A
        A = next(self.linearizer.linear_network.parameters())
        exp_A = torch.matrix_exp(A)

        # Jump forward
        z0_flat = z0.view(batch_size, -1)
        z1_pred_flat = torch.matmul(exp_A, z0_flat.T).T
        z1_pred = z1_pred_flat.view_as(x_noise)

        return {
            "jump_pred_std": z1_pred.std().item(),
            "jump_pred_max": z1_pred.max().item()
        }

    @torch.no_grad()
    def run_diagnostics(self, x_real, step):
        """
        Master method: Runs all tests and logs to wandb
        """
        self.linearizer.eval()
        device = x_real.device
        batch_size = x_real.shape[0]

        # Prepare Inputs
        t_rand = torch.rand(batch_size, device=device)
        x_noise = torch.randn_like(x_real)

        # 1. Invertibility
        inv_metrics = self._test_invertibility(x_real, t_rand)

        # 2. Latent Stats (using the z from the invertibility test)
        z_rand = self.linearizer.g(x_real, t_rand)
        if isinstance(z_rand, tuple): z_rand = z_rand[0]
        lat_metrics = self._test_latent_space(z_rand)

        # 3. Time Sensitivity
        time_metrics = self._test_time_sensitivity(x_real)

        # 4. Matrix Stability
        mat_metrics = self._test_matrix_stability(x_noise, device)

        # Log to WandB
        if wandb.run is not None:
            log_dict = {}
            for k, v in {**inv_metrics, **lat_metrics, **time_metrics, **mat_metrics}.items():
                log_dict[f"diagnostics/{k}"] = v
            wandb.log(log_dict, step=step)

        # Print summary to console for sanity check
        print(f"\n[Step {step}] Diagnostics:")
        print(f"  MSE: {inv_metrics['round_trip_mse']:.6f} | Max Err: {inv_metrics['round_trip_max_error']:.6f}")
        print(f"  Latent Std: {lat_metrics['latent_std']:.3f} | Jump Std: {mat_metrics['jump_pred_std']:.3f}")

        self.linearizer.train()





    @torch.no_grad()
    def _sample_exp(self, z0, t1):
        z1 = self.linearizer.linear_network.forward_exponential(z0)
        x_pred = self.linearizer.g_inverse(z1, t=t1)
        return x_pred

    @torch.no_grad()
    def _sample_discrete_collapsed(self, z0_flat, t1, step_matrix, num_steps, x_noise):

        collapsed_matrix = torch.linalg.matrix_power(step_matrix, num_steps)
        z1_collapsed_flat = torch.matmul(collapsed_matrix, z0_flat.T).T
        z1_collapsed = z1_collapsed_flat.view_as(x_noise)
        x_pred = self.linearizer.g_inverse(z1_collapsed, t=t1)
        if isinstance(x_pred, tuple): x_pred = x_pred[0]
        return x_pred

    @torch.no_grad()
    def _sample_iterative(self, z0_flat, t1, step_matrix, num_steps, x_noise):
        z_curr_flat = z0_flat.clone()
        for i in range(num_steps):
            z_curr_flat = torch.matmul(step_matrix, z_curr_flat.T).T

        z1_iter = z_curr_flat.view_as(x_noise)

        x_pred = self.linearizer.g_inverse(z1_iter, t=t1)
        if isinstance(x_pred, tuple): x_pred = x_pred[0]


        return x_pred

    @torch.no_grad()
    def sample(self, num_samples=16, img_size=IMG_SIZE, channels=IN_CHANNELS, num_steps=NUM_SAMPLING_STEPS, device=device):
        self.linearizer.eval()

        # Base Setup & Encoding
        x_noise = torch.randn(num_samples, channels, img_size, img_size, device=device)
        t0 = torch.zeros(num_samples, device=device)
        t1 = torch.ones(num_samples, device=device)

        z0 = self.linearizer.g(x_noise, t=t0)
        if isinstance(z0, tuple): z0 = z0[0]
        z0_flat = z0.view(num_samples, -1)

        # Linear Matrix Setup
        A = next(self.linearizer.linear_network.parameters())
        I = torch.eye(A.shape[0], device=device)
        delta_t = 1.0 / num_steps
        step_matrix = (I + delta_t*A)

        x_pred_exp = self._sample_exp(z0, t1) # exp sampling
        x_pred_discrete_collapsed = self._sample_discrete_collapsed(z0_flat, t1, step_matrix, num_steps, x_noise) # discrete collapsed matrix sampling
        x_pred_iterative = self._sample_iterative(z0_flat, t1, step_matrix, num_steps, x_noise) # iterative sampling

        self.linearizer.train()
        return x_pred_exp, x_pred_discrete_collapsed, x_pred_iterative
        # return {
        #     "exp": x_pred_exp,
        #     "discrete_collapsed": x_pred_discrete_collapsed,
        #     "iterative": x_pred_iterative
        # }


## 3. Training

Dataset loading, currently MNIST

In [ ]:
def get_optimized_loader(train_bs, val_bs, target_size):
    num_cpus = min(4, multiprocessing.cpu_count())
    transform = transforms.Compose([transforms.Resize(target_size), transforms.ToTensor()])

    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=train_bs, shuffle=True,
                              num_workers=num_cpus, pin_memory=True)
    return train_loader

train_loader = get_optimized_loader(BATCH_SIZE, VAL_BATCH_SIZE, IMG_SIZE)
print(f"Data loaded with {min(4, multiprocessing.cpu_count())} workers.")

Data loaded with 2 workers.


Function for showing given samples

In [ ]:
def show_samples(samples, title="Generated Samples"):
    samples = samples.detach().cpu().clamp(0, 1)
    grid_size = int(len(samples)**0.5)

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(6, 6))
    fig.suptitle(title)

    for i, ax in enumerate(axes.flat):
        if i < len(samples):
            ax.imshow(samples[i].permute(1, 2, 0).squeeze(), cmap='gray')
            ax.axis('off')
    plt.show()

Function for sampling, showing and logging samples to W&B

In [ ]:
# def sample_and_show(model, epoch, wandb, model_type_string="RAW", num_samples=16, img_size=IMAGE_SIZE, channels=IN_CHANNELS, device=device):
#     print(f'--- Epoch {epoch+1}: Sampling with {model_type_string} Model ---')
#     model.eval()
#     model_matcher = TimeG_FlowMatcher(model)
#     model_matcher.linearizer.linear_network.cache_exponential()

#     with torch.no_grad():
#         samples_exp, samples_discrete_collapsed, samples_iterative = model_matcher.sample(self, num_samples=num_samples, img_size=IMAGE_SIZE, channels=IN_CHANNELS, num_steps=NUM_SAMPLING_STEPS, device=device)(num_samples, img_size, channels, device)

#     title = f'Epoch {epoch+1} - {model_type_string}'

#     show_samples(samples_exp, title=f'{title} - Exp')
#     wandb_images_exp = [wandb.Image(img) for img in samples_exp]
#     wandb.log({f"samples/generated_images_{model_type_string}": wandb_images_raw, "epoch": epoch + 1})

#     show_samples(samples_discrete_collapsed, title=f'{title} - Discrete Collapse')
#     show_samples(samples_iterative, title=f'{title} - Iterative')



def sample_and_show(model, epoch, batch_precent, wandb_run, model_type_string="RAW", num_steps=NUM_SAMPLING_STEPS, num_samples=NUM_SAMPLES, img_size=IMG_SIZE, channels=IN_CHANNELS, device=device):
    print(f'Epoch: {epoch+1} - Batch Percent: {batch_precent} - Model Type: {model_type_string}')

    model.eval()
    model_matcher = TimeG_FlowMatcher(model)

    with torch.no_grad(): # Generate samples from each of the 3 sampling types
        samples_exp, samples_discrete_collapsed, samples_iter = model_matcher.sample(
            num_samples=num_samples,
            img_size=img_size,
            channels=channels,
            num_steps=num_steps,
            device=device
        )

    title_base = f'Epoch {epoch+1} - {model_type_string}'

    show_samples(samples_exp, title=f'{title_base} - Exact Matrix Exp')
    show_samples(samples_discrete_collapsed, title=f'{title_base} - Discrete Collapsed (Power)')
    show_samples(samples_iter, title=f'{title_base} - Iterative Euler (Loop)')

    # Log to W&B
    # We create grids for cleaner logging, but can also log individual lists
    if wandb_run is not None:
        # Create normalized grids
        grid_exp = vutils.make_grid(samples_exp.clamp(-1, 1), normalize=True, nrow=4)
        grid_col = vutils.make_grid(samples_discrete_collapsed.clamp(-1, 1), normalize=True, nrow=4)
        grid_iter = vutils.make_grid(samples_iter.clamp(-1, 1), normalize=True, nrow=4)

        wandb_run.log({
            f"samples/{model_type_string}_1_Exact_Exp": wandb.Image(grid_exp, caption=f"{model_type_string} EXP"),
            f"samples/{model_type_string}_2_Discrete_Collapsed": wandb.Image(grid_col, caption=f"{model_type_string} Collapsed"),
            f"samples/{model_type_string}_3_Iterative_Euler": wandb.Image(grid_iter, caption=f"{model_type_string} Iterative"),
            "epoch": epoch + 1
        }, step=epoch+1)

    model.train()

    return samples_exp, samples_discrete_collapsed, samples_iter

In [ ]:
def save_checkpoint(model, ema, optimizer, epoch, save_dir_raw=SAVE_DIR_RAW, save_dir_ema=SAVE_DIR_EMA):
    """
    Saves model weights, EMA weights, and a full training checkpoint.
    """
    # Ensure directories exist
    os.makedirs(save_dir_raw, exist_ok=True)
    os.makedirs(save_dir_ema, exist_ok=True)

    epoch_num = epoch + 1
    print(f"Saving models at Epoch {epoch_num}...")

    # 1. Save Raw Model Weights (Lightweight - for inference/transfer)
    raw_path = os.path.join(save_dir_raw, f'model_{epoch_num}.pth')
    torch.save(model.state_dict(), raw_path)

    # 2. Save EMA Model Weights (Best for sampling/generation)
    ema_path = os.path.join(save_dir_ema, f'model_ema_{epoch_num}.pth')
    torch.save(ema.get_model().state_dict(), ema_path)

    # 3. Save Full Checkpoint (For resuming training)
    # This contains optimizer state, allowing you to resume exactly where you left off
    checkpoint = {
        'epoch': epoch_num,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        # Optional: Save EMA internal state if your EMA class has buffers (shadow params)
        # 'ema_state_dict': ema.state_dict()
    }
    checkpoint_path = os.path.join(save_dir_raw, f'model_{epoch_num}_optimizer.pth')
    torch.save(checkpoint, checkpoint_path)

    print(f"✅ Models saved at Epoch {epoch_num}.")

# --- How to use it in your training loop ---
# save_checkpoint(model, ema, optimizer, epoch, SAVE_DIR_RAW, SAVE_DIR_EMA)

Preparation for training loop

In [ ]:
def training_init(
        img_size=IMG_SIZE,
        in_channels=IN_CHANNELS,
        num_layers_g=NUM_LAYERS_G,
        model_channels=MODEL_CHANNELS,
        ema_decay=EMA_DECAY,
        lambdas_dict=LAMBDAS_DICT,
        learning_rate=LR,
        model_name=MODEL_NAME,
        device=device,
    ):
    """
    Initializes the Model, EMA, Flow Matcher, and Optimizer.
    """
    # Define Architecture
    flat_dim = IN_CHANNELS * IMG_SIZE * IMG_SIZE
    fixed_A = FixedLinearMatrix(flat_dim).to(device)
    g_net = InvUnet(NUM_LAYERS_G, IN_CHANNELS, IMG_SIZE, creat_song_unet, MODEL_CHANNELS).to(device)

    # Create Linearized Model
    model = TimeVaryingGLinearizer(g_net, fixed_A).to(device)

    # Create EMA Wrapper
    ema = EMA(model, decay=EMA_DECAY)

    # Create Flow Matcher Loss Module
    fm = TimeG_FlowMatcher(
        model,
        lambdas_dict=lambdas_dict
    )

    # Create Optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    return model, ema, fm, optimizer

model, ema, fm, optimizer = training_init()

**Training loop**

In [ ]:
def train_model(
        model,
        ema,
        fm,
        optimizer,
        train_loader,
        wandb_run=None,
        num_epochs=EPOCHS,
        gradiend_clip_threshold=GRADIENT_CLIP_THRESOLD,
        save_dir_raw=SAVE_DIR_RAW,
        save_dir_ema=SAVE_DIR_EMA,
        device=device
):
    print("Starting Training")

    for epoch in range(num_epochs):
        model.train()

        num_batches = len(train_loader)
        milestones = {
            num_batches // 4: "25%",
            num_batches // 2: "50%",
            (3 * num_batches) // 4: "75%",
            num_batches : "100%"
        }


        with tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch") as pbar:
            for batch_idx, (data, _) in enumerate(pbar):
                data = data.to(device)
                optimizer.zero_grad()

                # loss, fm_loss, lpips_fm_loss, target_L2_loss, target_LPIPS_loss, iso_loss, spec_loss, frob_loss, vel_loss = fm.training_losses(data)
                losses_dict = fm.training_losses(data)

                loss = losses_dict["loss/total"]
                loss.backward()

                gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradiend_clip_threshold) # We do clipping, but the returned gradient norm is before clipping

                optimizer.step()

                # Update EMA
                ema.update(model)

                # W&B logging
                if wandb is not None:
                    wandb_run.log({
                        "train/gradient_norm": gradient_norm,
                        "epoch": epoch + 1,
                        "batch": batch_idx + (epoch * len(train_loader)), # Absolute step count
                        **losses_dict
                    })


                # if reached 25% batches
                if batch_idx in milestones:
                    batch_percent_str = milestones[batch_idx]

                    samples_exp_raw, samples_discrete_collapsed_raw, samples_iter_raw = sample_and_show(model, epoch, batch_percent_str, wandb_run, model_type_string="RAW", device=device)
                    samples_exp_ema, samples_discrete_collapsed_ema, samples_iter_ema = sample_and_show(ema, epoch, batch_percent_str, wandb_run, model_type_string="EMA", device=device)

                    save_checkpoint(model, ema, optimizer, epoch, SAVE_DIR_RAW, SAVE_DIR_EMA)

                    model.run_diagnostics(samples_exp_raw, epoch)
                    # ema.run_diagnostics(samples_exp_ema, epoch)

                # pbar.set_postfix(loss=f"{loss.item():.4f}", fm=f"{fm_loss:.4f}", lpips_fm_loss=f"{lpips_fm_loss:.4f}", target_L2_loss=f"{target_L2_loss:.4f}", target_LPIPS_loss=f"{target_LPIPS_loss:.4f}", iso=f"{iso_loss:.4f}", spec=f"{spec_loss:.4f}", frob=f"{frob_loss:.4f}", vel=f"{vel_loss:.4f}", grad_norm = f"{gradient_norm:.4f}")
                pbar.set_postfix({k.replace("loss/", ""): f"{v:.4f}" for k, v in losses_dict.items()}, grad=f"{gradient_norm:.4f}")



        # print(f"Saving models at Epoch {epoch+1}...")
        # raw_model_path = os.path.join(SAVE_DIR_RAW, f'model_{epoch+1}.pth')
        # ema_model_path = os.path.join(SAVE_DIR_EMA, f'model_ema_{epoch+1}.pth')
        # torch.save(model.state_dict(), raw_model_path)
        # ema_inference_model = ema.get_model()
        # torch.save(ema_inference_model.state_dict(), ema_model_path)

        # # save optimizer too
        # checkpoint = {
        #         'epoch': epoch + 1,                            # Save the next epoch number
        #         'model_state_dict': model.state_dict(),        # Save the model weights
        #         'optimizer_state_dict': optimizer.state_dict() # Save the optimizer momentum/velocity
        #     }
        # raw_model_path = os.path.join(SAVE_DIR_RAW, f'model_{epoch+1}_with_optimizer.pth')
        # # Save the dictionary to a file
        # torch.save(checkpoint, raw_model_path)
        # print(f"Models saved at Epoch {epoch+1}.")


        # if (epoch + 1) % EVAL_INTERVAL == 0:
        #         # --------------------------------------------------
        #         # 1. RAW MODEL SAMPLING & LOGGING
        #         # --------------------------------------------------
        #         print(f"--- Epoch {epoch+1}: Sampling with RAW Model ---")
        #         model.eval()
        #         model_matcher = TimeG_FlowMatcher(model)
        #         model_matcher.linearizer.linear_network.cache_exponential()

        #         with torch.no_grad(): # Don't forget this to save memory during sampling!
        #             samples_raw = model_matcher.sample_exponential(16, IMG_SIZE, IN_CHANNELS, device)
        #         show_samples(samples_raw, title=f"Epoch {epoch+1} (Original)")

        #         # Convert to W&B Images and log
        #         if wandb_run is not None:
        #             wandb_images_raw = [wandb_run.Image(img) for img in samples_raw]
        #             wandb_run.log({"samples/generated_images_RAW": wandb_images_raw, "epoch": epoch + 1})

        #         # --------------------------------------------------
        #         # 2. EMA MODEL SAMPLING & LOGGING
        #         # --------------------------------------------------
        #         print(f"--- Epoch {epoch+1}: Sampling with EMA Model ---")
        #         ema_model = ema.get_model()
        #         ema_matcher = TimeG_FlowMatcher(ema_model)
        #         ema_model.linear_network.cache_exponential()

        #         with torch.no_grad(): # Don't forget this!
        #             samples_ema = ema_matcher.sample_exponential(16, IMG_SIZE, IN_CHANNELS, device)
        #         show_samples(samples_ema, title=f"Epoch {epoch+1} (EMA)")

        #         # Convert to W&B Images and log
        #         if wandb_run is not None:
        #             wandb_images_ema = [wandb_run.Image(img) for img in samples_ema]
        #             wandb_run.log({"samples/generated_images_EMA": wandb_images_ema, "epoch": epoch + 1})

    if wandb_run is not None:
        wandb_run.finish()

train_model(
    model=model,
    ema=ema,
    fm=fm,
    optimizer=optimizer,
    train_loader=train_loader,
    wandb_run=wandb_run
)

In [ ]:
# # save optimizer too
# checkpoint = {
#         'epoch': epoch + 1,                            # Save the next epoch number
#         'model_state_dict': model.state_dict(),        # Save the model weights
#         'optimizer_state_dict': optimizer.state_dict() # Save the optimizer momentum/velocity
#     }
# raw_model_path = os.path.join(SAVE_DIR_RAW, f'model_{epoch+1}_with_optimizer.pth')
# # Save the dictionary to a file
# torch.save(checkpoint, raw_model_path)

In [1]:
# print("Starting Training")

# for epoch in range(EPOCHS):
#     model.train()

#     num_batches = len(train_loader)
#     milestones = {
#         num_batches // 4: "25%",
#         num_batches // 2: "50%",
#         (3 * num_batches) // 4: "75%"
#     }


#     with tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch") as pbar:
#         for batch_idx, (data, _) in enumerate(pbar):
#             data = data.to(device)
#             optimizer.zero_grad()

#             loss, fm_loss, lpips_fm_loss, target_L2_loss, target_LPIPS_loss, iso_loss, spec_loss, frob_loss, vel_loss = fm.training_losses(data)

#             loss.backward()

#             # gradient_norm = g(model.parameters(), max_norm=GRADIENT_CLIP_THRESOLD)
#             gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP_THRESOLD)

#             optimizer.step()

#             # Update EMA
#             ema.update(model)

#             # W&B logging
#             wandb.log({
#                 "train/total_loss": loss.item(),
#                 "train/fm_loss": fm_loss,
#                 "lpips_loss": lpips_fm_loss,
#                 "target_L2_loss": target_L2_loss,
#                 "target_LPIPS_loss": target_LPIPS_loss,
#                 "train/iso_loss": iso_loss,
#                 "spec_loss": spec_loss,
#                 "frob_loss": frob_loss,
#                 "train/vel_loss": vel_loss,
#                 "train/gradient_norm": gradient_norm,
#                 "epoch": epoch + 1,
#                 "batch": batch_idx + (epoch * len(train_loader)), # Absolute step count
#             })


#             # if reached 25% batches
#             if batch_idx in milestones:
#                 percent_str = milestones[batch_idx]

#                 print(f"--- Epoch {epoch+1}, Batches processed: {percent_str}, Sampling with RAW Model ---")
#                 model.eval()
#                 model_matcher = TimeG_FlowMatcher(model)
#                 model_matcher.linearizer.linear_network.cache_exponential()
#                 samples = model_matcher.sample_exponential(16, IMG_SIZE, IN_CHANNELS, device)
#                 show_samples(samples, title=f"Epoch {epoch+1} (RAW)")

#                 print(f"--- Epoch {epoch+1}, Batches processed: {percent_str}, Sampling with EMA Model ---")
#                 ema_model = ema.get_model()
#                 ema_matcher = TimeG_FlowMatcher(ema_model)
#                 ema_model.linear_network.cache_exponential()
#                 samples = ema_matcher.sample_exponential(16, IMG_SIZE, IN_CHANNELS, device)
#                 show_samples(samples, title=f"Epoch {epoch+1} (EMA)")



#             pbar.set_postfix(loss=f"{loss.item():.4f}", fm=f"{fm_loss:.4f}", lpips_fm_loss=f"{lpips_fm_loss:.4f}", target_L2_loss=f"{target_L2_loss:.4f}", target_LPIPS_loss=f"{target_LPIPS_loss:.4f}", iso=f"{iso_loss:.4f}", spec=f"{spec_loss:.4f}", frob=f"{frob_loss:.4f}", vel=f"{vel_loss:.4f}", grad_norm = f"{gradient_norm:.4f}")




#     print(f"Saving models at Epoch {epoch+1}...")
#     raw_model_path = os.path.join(SAVE_DIR_RAW, f'model_{epoch+1}.pth')
#     ema_model_path = os.path.join(SAVE_DIR_EMA, f'model_ema_{epoch+1}.pth')
#     torch.save(model.state_dict(), raw_model_path)
#     ema_inference_model = ema.get_model()
#     torch.save(ema_inference_model.state_dict(), ema_model_path)

#     # save optimizer too
#     checkpoint = {
#             'epoch': epoch + 1,                            # Save the next epoch number
#             'model_state_dict': model.state_dict(),        # Save the model weights
#             'optimizer_state_dict': optimizer.state_dict() # Save the optimizer momentum/velocity
#         }
#     raw_model_path = os.path.join(SAVE_DIR_RAW, f'model_{epoch+1}_with_optimizer.pth')
#     # Save the dictionary to a file
#     torch.save(checkpoint, raw_model_path)
#     print(f"Models saved at Epoch {epoch+1}.")

#     # if (epoch + 1) % EVAL_INTERVAL == 0:
#     #     print(f"--- Epoch {epoch+1}: Sampling with EMA Model ---")
#     #     ema_model = ema.get_model()
#     #     ema_matcher = TimeG_FlowMatcher(ema_model)
#     #     ema_model.linear_network.cache_exponential()
#     #     samples = ema_matcher.sample_exponential(16, IMG_SIZE, IN_CHANNELS, device)
#     #     show_samples(samples, title=f"Epoch {epoch+1} (EMA)")
#     # if (epoch + 1) % EVAL_INTERVAL == 0:
#     #     print(f"--- Epoch {epoch+1}: Sampling with RAW Model ---")
#     #     model.eval()
#     #     model_matcher = TimeG_FlowMatcher(model)
#     #     model_matcher.linearizer.linear_network.cache_exponential()
#     #     samples = model_matcher.sample_exponential(16, IMG_SIZE, IN_CHANNELS, device)
#     #     show_samples(samples, title=f"Epoch {epoch+1} (Original)")




#     if (epoch + 1) % EVAL_INTERVAL == 0:
#             # --------------------------------------------------
#             # 1. RAW MODEL SAMPLING & LOGGING
#             # --------------------------------------------------
#             print(f"--- Epoch {epoch+1}: Sampling with RAW Model ---")
#             model.eval()
#             model_matcher = TimeG_FlowMatcher(model)
#             model_matcher.linearizer.linear_network.cache_exponential()

#             with torch.no_grad(): # Don't forget this to save memory during sampling!
#                 samples_raw = model_matcher.sample_exponential(16, IMG_SIZE, IN_CHANNELS, device)
#             show_samples(samples_raw, title=f"Epoch {epoch+1} (Original)")

#             # Convert to W&B Images and log
#             wandb_images_raw = [wandb.Image(img) for img in samples_raw]
#             wandb.log({"samples/generated_images_RAW": wandb_images_raw, "epoch": epoch + 1})

#             # --------------------------------------------------
#             # 2. EMA MODEL SAMPLING & LOGGING
#             # --------------------------------------------------
#             print(f"--- Epoch {epoch+1}: Sampling with EMA Model ---")
#             ema_model = ema.get_model()
#             ema_matcher = TimeG_FlowMatcher(ema_model)
#             ema_model.linear_network.cache_exponential()

#             with torch.no_grad(): # Don't forget this!
#                 samples_ema = ema_matcher.sample_exponential(16, IMG_SIZE, IN_CHANNELS, device)
#             show_samples(samples_ema, title=f"Epoch {epoch+1} (EMA)")

#             # Convert to W&B Images and log
#             wandb_images_ema = [wandb.Image(img) for img in samples_ema]
#             wandb.log({"samples/generated_images_EMA": wandb_images_ema, "epoch": epoch + 1})



# wandb.finish()

In [ ]:
# # Initialize Model
# flat_dim = IN_CHANNELS * IMG_SIZE * IMG_SIZE
# fixed_A = FixedLinearMatrix(flat_dim).to(device) # Uses new Zero Init
# g_net = InvUnet(NUM_LAYERS_G, IN_CHANNELS, IMG_SIZE, creat_song_unet, MODEL_CHANNELS).to(device)
# model = TimeVaryingGLinearizer(g_net, fixed_A).to(device)

# # Initialize EMA
# ema = EMA(model, decay=EMA_DECAY)

# # Initialize Matcher (Disable Isometry, Enable Velocity Reg)
# fm = TimeG_FlowMatcher(model, lambda_iso=LAMBDA_ISO, lambda_frob=LAMBDA_FROB, lambda_vel=LAMBDA_VEL)

# optimizer = optim.Adam(model.parameters(), lr=LR)

# model.train()
# print(f"Saving models at Epoch 0...")
# raw_model_path = os.path.join(SAVE_DIR_RAW, f'model_0.pth')
# ema_model_path = os.path.join(SAVE_DIR_EMA, f'model_ema_0.pth')
# torch.save(model.state_dict(), raw_model_path)
# ema_inference_model = ema.get_model()
# torch.save(ema_inference_model.state_dict(), ema_model_path)
# print(f"Models saved at Epoch 0.")

# # Initialize W&B run
# wandb.init(
#     project="linearized-flow-matching", # project name
#     name=MODEL_NAME, # run name
#     config={
#         "epochs": EPOCHS,
#         "batch_size": BATCH_SIZE,
#         "learning_rate": LR,
#         "gradient_clip": GRADIENT_CLIP_THRESOLD,
#         "optimizer": "AdamW",
#         "lambda_fm": LAMBDA_FM_L2,
#         "lambda_fm_lpips": LAMBDA_FM_LPIPS,
#         "lambda_iso": LAMBDA_ISO,
#         "lambda_spectral": LAMBDA_SPEC,
#         "lambda_frob": LAMBDA_FROB,
#         "lambda_vel": LAMBDA_VEL,
#         "model_channels": MODEL_CHANNELS,
#         "num_layers_g": NUM_LAYERS_G,
#         "img_size": IMG_SIZE,
#         "in_channels": IN_CHANNELS,
#         # "inv_t_conf": INT_T_CONF,
#         "ema_decay": EMA_DECAY,
#         "lambda_target_l2": LAMBDA_TARGET_L2,
#         "lambda_target_lpips": LAMBDA_TARGET_LPIPS,

#     }
# )
